# EFGP-SING vs SING-SparseGP: Updated Benchmark Experiments

This notebook uses the current stable benchmark protocol with the **new exact-Cholesky M-step** and **frozen oracle emissions** (C = C_true).

## Why frozen oracle emissions?

When emissions C are learned from scratch, both methods get pulled into a basin determined by the SVD-init C, which differs from C_true by ~140% in Frobenius norm on these benchmarks. The lock-in attractor we used to see at (ℓ=2.29, σ²=3.42) on Duffing was the GT MLE *in the wrong latent frame* — a SING-wide identifiability artifact, not an EFGP issue.

With C frozen at C_true, both methods converge near the smoother-free GT MLE on all three benchmarks, and the comparison is on drift inference alone.

## Recent updates (2026-05-03 sanity-check session)

- **EFGP M-step uses dense Cholesky** instead of Hutchinson trace estimation. Single factorization gives both `log|A|` (from `diag(L)`) and `μ = A⁻¹h` (via two triangular solves). Cost is O(M³/3) per Adam step + O(M²) one-time `T_mat` build via direct fill from the BTTB conv vector.  Practical for `M ≲ 2000-4000`.
- The previous Hutchinson trace estimator at `n_hutchinson_mstep=4` was too high-variance on this problem (sign-changing `dws/dlogℓ` gives substantial off-diagonal mass in `A⁻¹ ∂A`). At fixed CG tol it took `n=64+` probes to match exact slogdet.
- **Default emission learning is OFF here** (`learn_emissions=False`, `update_R=False`); both `op` and `ip` are initialized using the synthetic ground-truth C.
- **SparseGP LR matched to EFGP at 1e-2**, made possible by Cholesky-stable gradients in `sing/sde.py` (replaced `jnp.linalg.inv(Kzz)` with `jla.cho_solve` in `f`, `ff`, `dfdx`, `update_dynamics_params`, `prior_term`, `get_posterior_*`) plus jitter bumped 1e-4 → 1e-3. The fix that mattered most was rewriting `prior_term` with explicit Cholesky-based KL — TFD's `MultivariateNormalFullCovariance` was failing on a barely-PSD `q_u_sigma` (sandwich product `Kzz @ A^{-1} @ Kzz` from `update_dynamics_params`).

The cell at the bottom plots **three landscapes** — ground-truth `-log p(x_true; θ)`, EFGP collapsed log-marginal, and SparseGP −ELBO — with both methods' EM trajectories overlaid on each, plus per-iter own-objective curves.


In [ ]:
import sys
from pathlib import Path
_SING = Path('/Users/colecitrenbaum/Documents/GPs/sing')
if str(_SING) not in sys.path:
    sys.path.insert(0, str(_SING))

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import time
import matplotlib.pyplot as plt
import matplotlib.gridspec as mgs

from sing.likelihoods import Gaussian
from sing.simulate_data import simulate_sde, simulate_gaussian_obs
from sing.sde import SparseGP
from sing.kernels import RBF
from sing.expectation import GaussHermiteQuadrature
from sing.sing import fit_variational_em
from sing.initialization import initialize_zs
import sing.efgp_em as em
import sing.efgp_jax_primitives as jp
import sing.efgp_jax_drift as jpd

print('jax devices:', jax.devices())

In [ ]:
def _make_data(drift_fn, x0, T, t_max, sigma, N_obs, seed, D=2):
    """Returns xs, ys, C_true. C_true is the synthetic ground-truth emission."""
    xs = simulate_sde(jr.PRNGKey(seed), x0=x0, f=drift_fn,
                      t_max=t_max, n_timesteps=T,
                      sigma=lambda x, t: sigma * jnp.eye(D))
    rng = np.random.default_rng(seed)
    C_true = jnp.asarray(rng.standard_normal((N_obs, D)) * 0.5)
    out_true = dict(C=C_true, d=jnp.zeros(N_obs),
                    R=jnp.full((N_obs,), 0.05))
    ys = simulate_gaussian_obs(jr.PRNGKey(seed + 1), xs, out_true)
    return np.asarray(xs), ys, C_true


def _shared_init(ys, D, N_obs, t_max, T, *, oracle_C=None):
    """If oracle_C is given, use it directly (frozen oracle frame); otherwise
    initialize C from the SVD of the centered observations."""
    if oracle_C is not None:
        op = dict(C=oracle_C, d=jnp.zeros(N_obs),
                   R=jnp.full((N_obs,), 0.1))
    else:
        yc = ys - ys.mean(0)
        _, _, Vt = jnp.linalg.svd(yc, full_matrices=False)
        op = dict(C=Vt[:D].T, d=ys.mean(0), R=jnp.full((N_obs,), 0.1))
    ip = jax.tree_util.tree_map(
        lambda x: x[None], dict(mu0=jnp.zeros(D), V0=jnp.eye(D) * 0.1))
    t_grid = jnp.linspace(0., t_max, T)
    return op, ip, t_grid


def _default_x_template(ip_init, T):
    diag_std0 = jnp.sqrt(jnp.diag(jnp.asarray(ip_init['V0'][0])))
    half_span = jnp.maximum(jnp.full_like(diag_std0, 3.0), 4.0 * diag_std0)
    return (jnp.asarray(ip_init['mu0'][0])[None, :]
            + jnp.linspace(-1., 1., max(T, 16))[:, None] * half_span[None, :])


def predict_obs_rmse(m_jax, output_params, ys):
    m = np.asarray(m_jax)
    C = np.asarray(output_params['C'])
    d = np.asarray(output_params['d'])
    y_hat = m @ C.T + d
    return float(np.sqrt(np.mean((y_hat - np.asarray(ys)) ** 2)))


def procrustes_align(m_inf, m_true):
    Xi, Xt = np.asarray(m_inf), np.asarray(m_true)
    bi, bt = Xi.mean(0), Xt.mean(0)
    A_T, *_ = np.linalg.lstsq(Xi - bi, Xt - bt, rcond=None)
    A = A_T.T
    return A, bt - A @ bi


# Frozen-emissions / oracle config: emissions identifiability is sidestepped
# by initializing C at the synthetic ground truth and not learning C/d/R.
def make_efgp_fit_cfg(kind, n_em):
    return dict(
        lengthscale=0.7,  # was 1.5; smaller init since we want to see EFGP
                          # converge from below toward GT MLE on Duffing
        variance=1.0,
        n_em_iters=n_em,
        n_estep_iters=10,
        rho_sched=jnp.linspace(0.05, 0.7, n_em),
        learn_emissions=False,    # frozen oracle C
        update_R=False,
        emission_warmup_iters=0,  # not needed when emissions frozen
        learn_kernel=True,
        n_mstep_iters=4,
        mstep_lr=0.01,
        n_hutchinson_mstep=4,     # ignored on the dense Cholesky path
        kernel_warmup_iters=8,
        eps_grid=1e-3,
        qf_nufft_eps=1e-7,
        qf_cg_tol=1e-5,
        S_marginal=2,
        tailored_grid_every_iter=False,
        K_per_dim=5,
        protocol=('frozen oracle C, dense Cholesky M-step, '
                  'adaptive-h K=5, eps=1e-3'),
    )


def make_sparse_fit_cfg(kind, n_em):
    rho = jnp.linspace(0.05, 0.7, n_em)
    return dict(
        init_lengthscale=0.7,
        init_output_scale=1.0,
        rho_sched=rho,
        n_iters_m=4,
        learning_rate=1e-2 * jnp.ones(n_em),  # matched to EFGP mstep_lr; stable
                                                # with Cholesky-based gradient + jitter=1e-3 (sde.py)
        zs_lim=2.5, num_per_dim=8,
        learn_emissions=False,    # frozen oracle C
        protocol='frozen oracle C, lr=1e-2 (matched to EFGP); Cholesky-stable',
    )


def run_efgp(lik, t_grid, op_init, ip_init, D, sigma, fit_cfg):
    x_template = _default_x_template(ip_init, t_grid.shape[0])
    t0 = time.time()
    mp, _, op, _, hist = em.fit_efgp_sing_jax(
        likelihood=lik, t_grid=t_grid,
        output_params=op_init, init_params=ip_init, latent_dim=D,
        lengthscale=fit_cfg['lengthscale'], variance=fit_cfg['variance'],
        sigma=sigma, sigma_drift_sq=sigma**2,
        eps_grid=fit_cfg['eps_grid'], S_marginal=fit_cfg['S_marginal'],
        n_em_iters=fit_cfg['n_em_iters'],
        n_estep_iters=fit_cfg['n_estep_iters'],
        rho_sched=fit_cfg['rho_sched'],
        learn_emissions=fit_cfg['learn_emissions'],
        update_R=fit_cfg['update_R'],
        emission_warmup_iters=fit_cfg['emission_warmup_iters'],
        learn_kernel=fit_cfg['learn_kernel'],
        n_mstep_iters=fit_cfg['n_mstep_iters'],
        mstep_lr=fit_cfg['mstep_lr'],
        n_hutchinson_mstep=fit_cfg['n_hutchinson_mstep'],
        kernel_warmup_iters=fit_cfg['kernel_warmup_iters'],
        tailored_grid_every_iter=fit_cfg['tailored_grid_every_iter'],
        K_per_dim=fit_cfg['K_per_dim'],
        X_template=x_template,
        qf_cg_tol=fit_cfg['qf_cg_tol'],
        qf_nufft_eps=fit_cfg['qf_nufft_eps'],
        verbose=False,
    )
    wall = time.time() - t0
    return dict(
        mp=mp, op=op, hist=hist, wall=wall,
        ls=float(hist.lengthscale[-1]),
        var=float(hist.variance[-1]),
        ls_path=np.array(hist.lengthscale),
        var_path=np.array(hist.variance),
        alpha_path=np.array(hist.variance) / np.maximum(np.array(hist.lengthscale)**2, 1e-12),
        obs_rmse=predict_obs_rmse(mp['m'][0], op, lik.ys_obs[0]),
        mean_R=float(np.mean(np.asarray(op['R']))),
        x_template=x_template,
        eps_grid=fit_cfg['eps_grid'],
        qf_nufft_eps=fit_cfg['qf_nufft_eps'],
        qf_cg_tol=fit_cfg['qf_cg_tol'],
        tailored_grid_every_iter=fit_cfg['tailored_grid_every_iter'],
        K_per_dim=fit_cfg['K_per_dim'],
        protocol=fit_cfg['protocol'],
    )


def run_sparsegp(lik, t_grid, op_init, ip_init, D, sigma, fit_cfg):
    quad = GaussHermiteQuadrature(D=D, n_quad=5)
    zs = initialize_zs(D=D, zs_lim=fit_cfg['zs_lim'],
                       num_per_dim=fit_cfg['num_per_dim'])
    sd = SparseGP(zs=zs, kernel=RBF(latent_dim=D), expectation=quad)
    sp0 = dict(length_scales=jnp.full((D,), fit_cfg['init_lengthscale']),
               output_scale=jnp.asarray(fit_cfg['init_output_scale']))
    dp_hist = []
    t0 = time.time()
    mp, nat_p, gp_post, dp, ip_final, op, ie_final, _ = fit_variational_em(
        key=jr.PRNGKey(33), fn=sd, likelihood=lik, t_grid=t_grid,
        drift_params=sp0, init_params=ip_init, output_params=op_init,
        sigma=sigma, rho_sched=fit_cfg['rho_sched'],
        n_iters=fit_cfg['rho_sched'].shape[0],
        n_iters_e=10, n_iters_m=fit_cfg['n_iters_m'],
        perform_m_step=True,
        learn_output_params=fit_cfg['learn_emissions'],
        learning_rate=fit_cfg['learning_rate'],
        print_interval=999, drift_params_history=dp_hist,
    )
    wall = time.time() - t0
    ls_path = np.array([float(np.mean(h['length_scales'])) for h in dp_hist])
    var_path = np.array([float(h['output_scale'])**2 for h in dp_hist])
    return dict(
        mp=mp, op=op, sd=sd, gp_post=gp_post, dp=dp, wall=wall,
        nat_p=nat_p, ip_final=ip_final, ie_final=ie_final,
        ls=float(jnp.mean(dp['length_scales'])),
        var=float(dp['output_scale'])**2,
        ls_path=ls_path, var_path=var_path,
        alpha_path=var_path / np.maximum(ls_path**2, 1e-12),
        obs_rmse=predict_obs_rmse(mp['m'][0], op, lik.ys_obs[0]),
        mean_R=float(np.mean(np.asarray(op['R']))),
        protocol=fit_cfg['protocol'],
    )


def eval_efgp_drift(efgp_res, t_grid, D, sigma, grid_pts_inferred):
    mp = efgp_res['mp']
    ls, var = efgp_res['ls'], efgp_res['var']
    x_template = efgp_res['x_template']
    x_min = np.asarray(x_template.min(axis=0))
    x_max = np.asarray(x_template.max(axis=0))
    x_extent = float((x_max - x_min).max())
    xcen = jnp.asarray(0.5 * (x_min + x_max), dtype=jnp.float32)
    if efgp_res['tailored_grid_every_iter']:
        grid = jp.spectral_grid_se(ls, var, x_template, eps=efgp_res['eps_grid'])
    else:
        grid = jp.spectral_grid_se_fixed_K(
            log_ls=jnp.log(jnp.asarray(ls, dtype=jnp.float32)),
            log_var=jnp.log(jnp.asarray(var, dtype=jnp.float32)),
            K_per_dim=efgp_res['K_per_dim'], X_extent=x_extent, xcen=xcen,
            d=D, eps=efgp_res['eps_grid'])
    m = jnp.asarray(mp['m'][0])
    S = jnp.asarray(np.asarray(mp['S'][0]))
    SS = jnp.asarray(np.asarray(mp['SS'][0]))
    del_t = t_grid[1:] - t_grid[:-1]
    mu_r, _, _ = jpd.compute_mu_r_jax(
        m, S, SS, del_t, grid, jr.PRNGKey(99),
        sigma_drift_sq=sigma**2, S_marginal=2, D_lat=D, D_out=D,
        cg_tol=efgp_res['qf_cg_tol'], nufft_eps=efgp_res['qf_nufft_eps'])
    Ef, _, _ = jpd.drift_moments_jax(
        mu_r, grid, jnp.asarray(grid_pts_inferred, dtype=jnp.float32),
        D_lat=D, D_out=D)
    return np.asarray(Ef)


def eval_sp_drift(sp_res, grid_pts_inferred):
    return np.asarray(sp_res['sd'].get_posterior_f_mean(
        sp_res['gp_post'], sp_res['dp'],
        jnp.asarray(grid_pts_inferred, dtype=jnp.float32)))


def _quiver(ax, GX, GY, F, color, title, xs_np=None):
    mag = np.linalg.norm(F, axis=1, keepdims=True).clip(1e-8)
    Fn = F / mag
    ax.quiver(GX, GY, Fn[:, 0].reshape(GX.shape), Fn[:, 1].reshape(GX.shape),
              color=color, angles='xy', scale_units='xy', scale=3.5, alpha=0.85,
              width=0.012, headwidth=3)
    if xs_np is not None:
        ax.plot(xs_np[:, 0], xs_np[:, 1], lw=0.6, color='red', alpha=0.3)
    ax.set_title(title, fontsize=9)
    ax.set_aspect('equal')
    ax.set_xlabel('$x_1$', fontsize=8)
    ax.set_ylabel('$x_2$', fontsize=8)
    ax.tick_params(labelsize=7)


In [ ]:
# NOTE: drift_fn must use jnp operations (JAX-traced by simulate_sde).
_A_rot = jnp.array([[-0.6, 1.0], [-1.0, -0.6]])

EXPS = [
    dict(
        name='Damped rotation',
        subtitle=r'$f(x)=Ax$, $A=[[-0.6,1],[-1,-0.6]]$ (linear)',
        drift_fn=lambda x, t: _A_rot @ x,
        x0=jnp.array([2.0, 0.0]),
        T=400, t_max=8.0, sigma=0.3, N_obs=8, seed=7,
        kind='linear', n_em=50,
        note='Linear row uses n_em=50 because the old n_em=25 budget was not enough for EFGP to settle.'
    ),
    dict(
        name='Duffing double-well',
        subtitle=r'$f=[x_2,\; x_1-x_1^3-0.5x_2]$ (two stable foci at $x_1=\pm1$)',
        drift_fn=lambda x, t: jnp.stack([x[1], x[0] - x[0]**3 - 0.5*x[1]]),
        x0=jnp.array([1.2, 0.0]),
        T=400, t_max=15.0, sigma=0.2, N_obs=8, seed=13,
        kind='nonlinear', n_em=20,
        note='SparseGP uses the conservative SING-demo schedule here; the original notebook settings could collapse σ².'
    ),
    dict(
        name='Anharmonic oscillator',
        subtitle=r'$f=[x_2,\; -x_1-0.3x_2-0.5x_1^3]$ (nonlinear single-well)',
        drift_fn=lambda x, t: jnp.stack([x[1], -x[0] - 0.3*x[1] - 0.5*x[0]**3]),
        x0=jnp.array([1.5, 0.0]),
        T=400, t_max=10.0, sigma=0.3, N_obs=8, seed=21,
        kind='nonlinear', n_em=20,
        note='SparseGP uses the conservative SING-demo schedule here; the old notebook settings could produce NaN hypers.'
    ),
]
D = 2

In [ ]:
all_results = []

for cfg in EXPS:
    print(f'\n=== {cfg["name"]} ===')
    print('  note:', cfg['note'])
    xs_np, ys, C_true = _make_data(cfg['drift_fn'], cfg['x0'], cfg['T'],
                                     cfg['t_max'], cfg['sigma'], cfg['N_obs'],
                                     cfg['seed'])
    op_init, ip_init, t_grid = _shared_init(
        ys, D, cfg['N_obs'], cfg['t_max'], cfg['T'], oracle_C=C_true)
    lik = Gaussian(ys[None], jnp.ones((1, cfg['T']), dtype=bool))

    efgp_fit = make_efgp_fit_cfg(cfg['kind'], cfg['n_em'])
    sparse_fit = make_sparse_fit_cfg(cfg['kind'], cfg['n_em'])

    lo = xs_np.min(0) - 0.4
    hi = xs_np.max(0) + 0.4
    g0 = np.linspace(lo[0], hi[0], 14)
    g1 = np.linspace(lo[1], hi[1], 14)
    GX, GY = np.meshgrid(g0, g1, indexing='ij')
    grid_pts = np.stack([GX.ravel(), GY.ravel()], axis=-1).astype(np.float32)
    f_true = np.array([np.asarray(cfg['drift_fn'](jnp.asarray(p), 0.))
                       for p in grid_pts])
    zero_rmse = float(np.sqrt(np.mean(f_true**2)))

    print('  EFGP...', end=' ', flush=True)
    efgp = run_efgp(lik, t_grid, op_init, ip_init, D, cfg['sigma'], efgp_fit)
    A_e, b_e = procrustes_align(efgp['mp']['m'][0], xs_np)
    lat_e = float(np.sqrt(np.mean(
        (np.asarray(efgp['mp']['m'][0]) @ A_e.T + b_e - xs_np)**2)))
    efgp['latent_rmse'] = lat_e
    grid_e = (grid_pts - b_e) @ np.linalg.inv(A_e).T
    f_efgp = eval_efgp_drift(efgp, t_grid, D, cfg['sigma'], grid_e) @ A_e.T
    efgp['drift_rmse'] = float(np.sqrt(np.mean((f_efgp - f_true)**2)))
    efgp['rel'] = efgp['drift_rmse'] / zero_rmse
    print(f'{efgp["wall"]:.1f}s  obs={efgp["obs_rmse"]:.4f}  ell={efgp["ls"]:.3f}  '
          f'lat={lat_e:.4f}  drift={efgp["drift_rmse"]:.4f} ({efgp["rel"]:.1%})')

    print('  SparseGP...', end=' ', flush=True)
    sp = run_sparsegp(lik, t_grid, op_init, ip_init, D, cfg['sigma'], sparse_fit)
    A_s, b_s = procrustes_align(sp['mp']['m'][0], xs_np)
    lat_s = float(np.sqrt(np.mean(
        (np.asarray(sp['mp']['m'][0]) @ A_s.T + b_s - xs_np)**2)))
    sp['latent_rmse'] = lat_s
    grid_s = (grid_pts - b_s) @ np.linalg.inv(A_s).T
    f_sp = eval_sp_drift(sp, grid_s) @ A_s.T
    sp['drift_rmse'] = float(np.sqrt(np.mean((f_sp - f_true)**2)))
    sp['rel'] = sp['drift_rmse'] / zero_rmse
    print(f'{sp["wall"]:.1f}s  obs={sp["obs_rmse"]:.4f}  ell={sp["ls"]:.3f}  '
          f'lat={lat_s:.4f}  drift={sp["drift_rmse"]:.4f} ({sp["rel"]:.1%})')

    all_results.append(dict(
        cfg=cfg, xs_np=xs_np, t_grid=t_grid,
        GX=GX, GY=GY, grid_pts=grid_pts,
        f_true=f_true, f_efgp=f_efgp, f_sp=f_sp,
        zero_rmse=zero_rmse, efgp=efgp, sp=sp,
        A_e=A_e, b_e=b_e, A_s=A_s, b_s=b_s,
        efgp_protocol=efgp_fit['protocol'],
        sparse_protocol=sparse_fit['protocol'],
    ))

print('\nDone.')


## Per-experiment results

Each row now uses the stable protocol for that regime. The summary table includes observation RMSE and the learned mean noise level `R`; the text block below it records the protocol choice for each method.

In [ ]:
for res in all_results:
    cfg = res['cfg']
    efgp, sp = res['efgp'], res['sp']
    GX, GY = res['GX'], res['GY']
    f_true, f_efgp, f_sp = res['f_true'], res['f_efgp'], res['f_sp']
    xs_np = res['xs_np']
    n_iters_efgp = len(efgp['ls_path'])
    n_iters_sp = len(sp['ls_path'])

    fig = plt.figure(figsize=(17, 8.2))
    gs = mgs.GridSpec(2, 5, figure=fig, hspace=0.48, wspace=0.38,
                      width_ratios=[1, 1, 1, 1.15, 1.15])

    for col, (title, F, col_c) in enumerate([
        ('True drift (direction)', f_true, '0.35'),
        (f'EFGP  RMSE={efgp["drift_rmse"]:.3f}', f_efgp, 'C0'),
        (f'SparseGP  RMSE={sp["drift_rmse"]:.3f}', f_sp, 'C2'),
    ]):
        ax = fig.add_subplot(gs[0, col])
        _quiver(ax, GX, GY, F, col_c, title, xs_np if col == 0 else None)

    iters_e = np.arange(n_iters_efgp)
    iters_s = np.arange(n_iters_sp)

    ax_ls = fig.add_subplot(gs[1, 0])
    ax_ls.plot(iters_e, efgp['ls_path'], color='C0', lw=1.8, label='EFGP')
    ax_ls.plot(iters_s, sp['ls_path'], color='C2', lw=1.8, ls='--', label='SparseGP')
    ax_ls.set_xlabel('EM iter', fontsize=8)
    ax_ls.set_ylabel('lengthscale ℓ', fontsize=8)
    ax_ls.set_title('ℓ trajectory', fontsize=9)
    ax_ls.legend(fontsize=7, loc='best')
    ax_ls.tick_params(labelsize=7)

    ax_var = fig.add_subplot(gs[1, 1])
    ax_var.plot(iters_e, efgp['var_path'], color='C0', lw=1.8, label='EFGP')
    ax_var.plot(iters_s, sp['var_path'], color='C2', lw=1.8, ls='--', label='SparseGP')
    ax_var.set_xlabel('EM iter', fontsize=8)
    ax_var.set_ylabel(r'$\sigma^2$ (variance)', fontsize=8)
    ax_var.set_title(r'$\sigma^2$ trajectory', fontsize=9)
    ax_var.legend(fontsize=7, loc='best')
    ax_var.tick_params(labelsize=7)

    ax_lat = fig.add_subplot(gs[1, 2])
    m_e = np.asarray(efgp['mp']['m'][0]) @ res['A_e'].T + res['b_e']
    m_s = np.asarray(sp['mp']['m'][0]) @ res['A_s'].T + res['b_s']
    ax_lat.plot(xs_np[:, 0], xs_np[:, 1], color='gray', lw=1.0, alpha=0.7, label='truth')
    ax_lat.plot(m_e[:, 0], m_e[:, 1], color='C0', lw=0.9, alpha=0.7,
                label=f'EFGP ({efgp["latent_rmse"]:.3f})')
    ax_lat.plot(m_s[:, 0], m_s[:, 1], color='C2', lw=0.9, alpha=0.7,
                label=f'SparseGP ({sp["latent_rmse"]:.3f})')
    ax_lat.set_title('Latent recovery (Procrustes)', fontsize=9)
    ax_lat.set_xlabel('$x_1$', fontsize=8)
    ax_lat.set_ylabel('$x_2$', fontsize=8)
    ax_lat.legend(fontsize=6, loc='best')
    ax_lat.tick_params(labelsize=7)
    ax_lat.set_aspect('equal')

    ax_txt = fig.add_subplot(gs[:, 3:])
    ax_txt.axis('off')
    rows = [
        ('', 'EFGP', 'SparseGP'),
        ('Wall (s)', f'{efgp["wall"]:.1f}', f'{sp["wall"]:.1f}'),
        ('Obs RMSE', f'{efgp["obs_rmse"]:.4f}', f'{sp["obs_rmse"]:.4f}'),
        ('Latent RMSE', f'{efgp["latent_rmse"]:.4f}', f'{sp["latent_rmse"]:.4f}'),
        ('Drift RMSE', f'{efgp["drift_rmse"]:.4f}', f'{sp["drift_rmse"]:.4f}'),
        ('Rel. drift RMSE', f'{efgp["rel"]:.1%}', f'{sp["rel"]:.1%}'),
        ('Final ℓ', f'{efgp["ls"]:.3f}', f'{sp["ls"]:.3f}'),
        ('Final σ²', f'{efgp["var"]:.3f}', f'{sp["var"]:.3f}'),
        ('mean R', f'{efgp["mean_R"]:.4f}', f'{sp["mean_R"]:.4f}'),
        ('EM iters', str(cfg['n_em']), str(cfg['n_em'])),
        ('zero-drift RMSE', f'{res["zero_rmse"]:.4f}', ''),
        ('T', str(cfg['T']), f't_max={cfg["t_max"]}'),
        ('σ_diff', str(cfg['sigma']), ''),
    ]
    col_w = [0.38, 0.31, 0.31]
    y = 0.97
    for r in rows:
        x = 0.02
        for j, (txt, w) in enumerate(zip(r, col_w)):
            weight = 'bold' if (r[0] == '' and y > 0.92) else 'normal'
            if j == 0:
                weight = 'bold'
            ax_txt.text(x, y, txt, transform=ax_txt.transAxes,
                        fontsize=9, va='top', fontfamily='monospace',
                        fontweight=weight)
            x += w
        y -= 0.065

    ax_txt.text(0.02, 0.18, 'Protocol note:', transform=ax_txt.transAxes,
                fontsize=9, fontweight='bold')
    ax_txt.text(0.02, 0.13, cfg['note'], transform=ax_txt.transAxes,
                fontsize=8.5, wrap=True)
    ax_txt.text(0.02, 0.07, f'EFGP: {res["efgp_protocol"]}', transform=ax_txt.transAxes,
                fontsize=8.2, wrap=True)
    ax_txt.text(0.02, 0.02, f'SparseGP: {res["sparse_protocol"]}', transform=ax_txt.transAxes,
                fontsize=8.2, wrap=True)

    fig.suptitle(f'{cfg["name"]}  —  {cfg["subtitle"]}',
                 fontsize=11, fontweight='bold')
    plt.show()

## Three-objective comparison: GT marginal, EFGP loss, SparseGP −ELBO

For each benchmark we compute **three** landscapes:
1. **Ground-truth `-log p(x_true; θ)`** — smoother-free oracle (only available for synthetic experiments). Dense GP marginal on `(xs[:-1], velocities)` with the *true* latent `xs` short-circuiting any smoother bias. This is the gold standard: the surface that should be approximated by any unbiased EM-with-smoother.
2. **EFGP collapsed log-marginal** `L(log ℓ, log σ²)` at the EFGP-converged smoother — the surface EFGP's M-step minimizes.
3. **SparseGP −ELBO** at the SparseGP-converged `q(x)` and inducing points — the surface SparseGP's M-step minimizes.

For each row (one benchmark), we overlay both methods' EM trajectories on each landscape, plus per-iter own-objective loss curves at the bottom. The trajectory endpoint gap on the GT landscape (Δ in legends) tells us how far each method ended up from the smoother-free MLE.

**Layout: 4 rows × 3 cols (3 benchmarks)**
- Row 0 (GT): smoother-free oracle. The trajectory's gap to the gold star (GT MLE) tells you how close the method got to what the data actually wants.
- Row 1: EFGP collapsed loss surface
- Row 2: SparseGP −ELBO surface
- Row 3: per-iter own-objective loss curves


In [ ]:
## Two-objective comparison: hyper trajectories on EFGP loss, SparseGP −ELBO,
## AND ground-truth −log p(x_true; θ)
#
# For each benchmark, we compute THREE landscapes:
#   (a) Ground-truth `-log p(x_true; θ)` — smoother-free oracle (synthetic only)
#       Dense GP marginal on (xs[:-1], velocities); known xs short-circuit
#       the smoother. This is the *target* both methods should approach.
#   (b) EFGP collapsed log-marginal at the EFGP-converged smoother
#   (c) SparseGP −ELBO at the SparseGP-converged state (frozen q(x) + inducing)
#
# Both methods' EM trajectories are overlaid on each landscape. Plus per-iter
# own-objective loss curves at the bottom.
#
# Layout: 4 rows × 3 cols
#   row 0: GT -log p(x_true; θ) — the only objective neither method directly
#          targets, but the one we'd hope they approximate
#   row 1: EFGP loss surface
#   row 2: SparseGP −ELBO surface
#   row 3: per-iter own-objective loss curves
#
# Cost: ~441 grid evals × 3 landscapes × 3 benchmarks ≈ 8-12 min total.

import os, math, time
import numpy as np
import jax, jax.numpy as jnp, jax.random as jr
import jax.scipy.linalg as _jla
from jax import vmap
import matplotlib.pyplot as plt
import scipy.linalg as _sla
import sing.efgp_jax_primitives as _jp
import sing.efgp_jax_drift as _jpd
from sing.efgp_jax_drift import _ws_real_se as _ws_real_se_fn
from sing.sing import compute_elbo_over_batch as _compute_elbo
from sing.utils.sing_helpers import natural_to_marginal_params as _nat_to_marg

K_FD = 5
LOG_LS_GRID = np.linspace(-2.0, 1.5, 21)
LOG_VAR_GRID = np.linspace(-2.5, 2.0, 21)


def _build_efgp_state(mp, t_grid, sigma, ls_pin, var_pin):
    X_template = (jnp.linspace(-3., 3., 16)[:, None] * jnp.ones((1, D)))
    X_extent = float((X_template.max(axis=0) - X_template.min(axis=0)).max())
    xcen = jnp.asarray(0.5 * (X_template.min(axis=0) + X_template.max(axis=0)),
                       dtype=jnp.float32)
    grid = _jp.spectral_grid_se_fixed_K(
        log_ls=jnp.log(jnp.asarray(ls_pin, dtype=jnp.float32)),
        log_var=jnp.log(jnp.asarray(var_pin, dtype=jnp.float32)),
        K_per_dim=K_FD, X_extent=X_extent, xcen=xcen, d=D, eps=1e-3)
    ms = mp['m'][0]; Ss = mp['S'][0]; SSs = mp['SS'][0]
    del_t = t_grid[1:] - t_grid[:-1]
    mu_r, _, top = _jpd.compute_mu_r_jax(
        ms, Ss, SSs, del_t, grid, jr.PRNGKey(99),
        sigma_drift_sq=sigma ** 2, S_marginal=8,
        D_lat=D, D_out=D, cg_tol=1e-9, max_cg_iter=4 * grid.M)
    ws_real0 = grid.ws.real.astype(grid.ws.dtype)
    ws_safe = jnp.where(jnp.abs(ws_real0) < 1e-30,
                        jnp.array(1e-30, dtype=ws_real0.dtype), ws_real0)
    A_init = _jp.make_A_apply(grid.ws, top, sigmasq=1.0)
    h0 = jax.vmap(A_init)(mu_r)
    z_r = h0 / ws_safe
    return grid, top, z_r


def _efgp_loss_at(grid, top, z_r, log_ls, log_var):
    M = int(grid.xis_flat.shape[0])
    cdtype = top.v_fft.dtype
    ws_real = _ws_real_se_fn(jnp.asarray(log_ls, dtype=jnp.float32),
                              jnp.asarray(log_var, dtype=jnp.float32),
                              grid.xis_flat, grid.h_per_dim[0], D)
    ws_c = ws_real.astype(cdtype)
    v_pad = jnp.fft.ifftn(top.v_fft).astype(cdtype)
    ns_v = tuple(2*n-1 for n in top.ns)
    v_conv = v_pad[tuple(slice(0, L) for L in ns_v)]
    d_ = len(top.ns)
    mi = jnp.indices(top.ns).reshape(d_, -1)
    offset = jnp.array([n-1 for n in top.ns], dtype=jnp.int32)
    diff = mi[:, :, None] - mi[:, None, :] + offset[:, None, None]
    T_mat = v_conv[tuple(diff[k] for k in range(d_))]
    eye_c = jnp.eye(M, dtype=cdtype)
    A = eye_c + ws_c[:, None] * T_mat * ws_c[None, :]
    L = jnp.linalg.cholesky(A)
    logdet = 2.0 * jnp.sum(jnp.log(jnp.diag(L).real))
    h = ws_c[None, :] * z_r
    def solve_one(b):
        y = _jla.solve_triangular(L, b, lower=True)
        return _jla.solve_triangular(L.conj().T, y, lower=False)
    mu = jax.vmap(solve_one)(h)
    det_loss = -0.5 * jnp.sum(jnp.real(jnp.sum(jnp.conj(h) * mu, axis=-1)))
    return float(det_loss + 0.5 * D * logdet)


def _build_sparse_neg_elbo_fn(sp_res, lik, t_grid, sigma):
    """Return a JIT'd fn (log_ls, log_var) -> -ELBO using frozen q(x) +
    frozen inducing pts from the SparseGP final state."""
    nat_p = sp_res['nat_p']
    mp = sp_res['mp']
    op = sp_res['op']
    ip_final = sp_res['ip_final']
    ie_final = sp_res['ie_final']
    sparse = sp_res['sd']
    inducing_frozen = sp_res['dp'].get('inducing_points', sparse.zs)
    T = lik.ys_obs.shape[1]
    trial_mask = jnp.ones((1, T), dtype=bool)
    _, ap = vmap(_nat_to_marg)(nat_p, trial_mask)
    inputs = jnp.zeros((1, T, 1))

    def neg_elbo(log_ls, log_var):
        drift_params = dict(
            length_scales=jnp.full((D,), jnp.exp(log_ls)),
            output_scale=jnp.exp(0.5 * log_var),
            inducing_points=inducing_frozen,
        )
        return -_compute_elbo(
            jr.PRNGKey(0), lik.ys_obs, lik.t_mask, trial_mask,
            sparse, lik, t_grid, drift_params,
            ip_final, op, nat_p, mp, ap, inputs, ie_final, sigma)[0]

    return jax.jit(neg_elbo)


def _gt_landscape(xs, sigma_drift, t_grid, LOG_LS, LOG_VAR):
    """-log p(x_true; θ) as a dense GP marginal on (xs[:-1], velocities).
    Smoother-free oracle for synthetic experiments where xs is known."""
    xs_np = np.asarray(xs)
    T_ = xs_np.shape[0]
    inputs = xs_np[:-1]
    dt = float(np.asarray(t_grid[1] - t_grid[0]))
    velocities = (xs_np[1:] - xs_np[:-1]) / dt
    n_obs = T_ - 1
    D_out = velocities.shape[1]
    diffs = inputs[:, None, :] - inputs[None, :, :]
    sq_dists = (diffs ** 2).sum(-1)
    noise_var = sigma_drift ** 2 / dt
    eye_n = np.eye(n_obs)
    L_grid = np.zeros((len(LOG_LS), len(LOG_VAR)))
    for i, ll in enumerate(LOG_LS):
        ell = math.exp(float(ll))
        K_unsc = np.exp(-0.5 * sq_dists / (ell ** 2))
        for j, lv in enumerate(LOG_VAR):
            var_ = math.exp(float(lv))
            A_ = var_ * K_unsc + noise_var * eye_n
            try:
                L_chol = np.linalg.cholesky(A_)
            except np.linalg.LinAlgError:
                L_grid[i, j] = np.nan
                continue
            logdet = 2.0 * np.log(np.diag(L_chol)).sum()
            quad = 0.0
            for d in range(D_out):
                z = _sla.solve_triangular(L_chol, velocities[:, d], lower=True)
                quad += float(z @ z)
            L_grid[i, j] = 0.5 * quad + 0.5 * D_out * logdet
    return L_grid


def _gt_loss_at(xs, sigma_drift, t_grid, log_ls, log_var):
    return float(_gt_landscape(xs, sigma_drift, t_grid,
                                np.array([log_ls]), np.array([log_var]))[0, 0])


def _adaptive_levels(L_norm, n=10):
    finite = L_norm[np.isfinite(L_norm) & (L_norm > 0)]
    if finite.size == 0:
        return [0.5, 1, 2, 5, 10, 20, 50, 100, 200, 500]
    lo, hi = float(finite.min()), float(finite.max())
    lo_use = max(lo, hi / 1000.0)
    return list(np.logspace(np.log10(lo_use), np.log10(hi), n))


def _render_panel(ax, LOG_LS, LOG_VAR, L_norm, e_ls, e_var, s_ls, s_var,
                   star_ll, star_lv, title, min_label, ls_init, var_init,
                   e_wall=None, s_wall=None, e_lat=None, s_lat=None,
                   e_endpt=None, s_endpt=None):
    levels = _adaptive_levels(L_norm)
    ax.contourf(LOG_LS, LOG_VAR, L_norm.T,
                 levels=[0]+levels, cmap='viridis_r', extend='max')
    cs = ax.contour(LOG_LS, LOG_VAR, L_norm.T,
                     levels=levels, colors='k', linewidths=0.4)
    ax.clabel(cs, inline=True, fontsize=6, fmt='%.2g')
    for log_a in [-2, -1, 0, 1, 2]:
        ax.plot(LOG_LS, LOG_LS*2 + log_a,
                color='white', linestyle=':', linewidth=0.5, alpha=0.5)
    el = 'EFGP traj'
    if e_wall is not None:
        el += f"  [{e_wall:.0f}s"
        if e_lat is not None: el += f", lat={e_lat:.3f}"
        if e_endpt is not None: el += f", Δ={e_endpt:.1f}"
        el += "]"
    sl = 'SparseGP traj'
    if s_wall is not None:
        sl += f"  [{s_wall:.0f}s"
        if s_lat is not None: sl += f", lat={s_lat:.3f}"
        if s_endpt is not None: sl += f", Δ={s_endpt:.1f}"
        sl += "]"
    ax.plot(np.log(e_ls), np.log(e_var), '-o', color='C0',
            markersize=3, linewidth=1.6, label=el)
    ax.plot(np.log(s_ls), np.log(s_var), '-s', color='C3',
            markersize=3, linewidth=1.4, label=sl)
    ax.scatter([np.log(ls_init)], [np.log(var_init)], marker='+', s=200,
                color='black', zorder=6, label='init')
    ax.scatter([star_ll], [star_lv], marker='*', s=240,
                color='gold', edgecolor='k', zorder=7, label=min_label)
    ax.set_xlim(LOG_LS.min(), LOG_LS.max())
    ax.set_ylim(LOG_VAR.min(), LOG_VAR.max())
    ax.set_xlabel('log ℓ', fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.legend(loc='lower right', fontsize=7)


# ---- Compute landscapes per benchmark ----
landscapes = []
for res in all_results:
    cfg = res['cfg']
    print(f"  computing landscapes for {cfg['name']}...", flush=True)
    sigma = cfg['sigma']
    op_init, ip_init, t_grid = _shared_init(
        np.asarray(_make_data(
            cfg['drift_fn'], cfg['x0'], cfg['T'], cfg['t_max'],
            cfg['sigma'], cfg['N_obs'], cfg['seed'])[1]),
        D, cfg['N_obs'], cfg['t_max'], cfg['T'])
    xs_np, ys = _make_data(cfg['drift_fn'], cfg['x0'], cfg['T'],
                            cfg['t_max'], cfg['sigma'], cfg['N_obs'],
                            cfg['seed'])
    lik = Gaussian(ys[None], jnp.ones((1, cfg['T']), dtype=bool))

    # EFGP loss landscape
    e = res['efgp']
    e_grid, e_top, e_zr = _build_efgp_state(
        e['mp'], t_grid, sigma, e['ls'], e['var'])
    L_efgp = np.zeros((len(LOG_LS_GRID), len(LOG_VAR_GRID)))
    t0 = time.time()
    for i, ll in enumerate(LOG_LS_GRID):
        for j, lv in enumerate(LOG_VAR_GRID):
            L_efgp[i, j] = _efgp_loss_at(e_grid, e_top, e_zr,
                                          float(ll), float(lv))
    eb = np.unravel_index(np.argmin(L_efgp), L_efgp.shape)
    eb_ll = float(LOG_LS_GRID[eb[0]]); eb_lv = float(LOG_VAR_GRID[eb[1]])
    L_efgp_min = float(L_efgp[eb])
    print(f"    EFGP: {time.time()-t0:.1f}s, min ℓ={math.exp(eb_ll):.2f}, "
          f"σ²={math.exp(eb_lv):.2f}", flush=True)

    # SparseGP -ELBO landscape
    sp = res['sp']
    neg_elbo_fn = _build_sparse_neg_elbo_fn(sp, lik, t_grid, sigma)
    L_sp = np.zeros((len(LOG_LS_GRID), len(LOG_VAR_GRID)))
    t0 = time.time()
    for i, ll in enumerate(LOG_LS_GRID):
        for j, lv in enumerate(LOG_VAR_GRID):
            L_sp[i, j] = float(neg_elbo_fn(
                jnp.asarray(ll, dtype=jnp.float32),
                jnp.asarray(lv, dtype=jnp.float32)))
    sb = np.unravel_index(np.argmin(L_sp), L_sp.shape)
    sb_ll = float(LOG_LS_GRID[sb[0]]); sb_lv = float(LOG_VAR_GRID[sb[1]])
    L_sp_min = float(L_sp[sb])
    print(f"    SparseGP -ELBO: {time.time()-t0:.1f}s, min ℓ={math.exp(sb_ll):.2f}, "
          f"σ²={math.exp(sb_lv):.2f}", flush=True)

    # Ground-truth landscape (smoother-free)
    t0 = time.time()
    L_gt = _gt_landscape(np.asarray(xs_np), sigma, t_grid,
                          LOG_LS_GRID, LOG_VAR_GRID)
    gb = np.unravel_index(np.nanargmin(L_gt), L_gt.shape)
    gb_ll = float(LOG_LS_GRID[gb[0]]); gb_lv = float(LOG_VAR_GRID[gb[1]])
    L_gt_min = float(L_gt[gb])
    print(f"    GT: {time.time()-t0:.1f}s, MLE ℓ={math.exp(gb_ll):.2f}, "
          f"σ²={math.exp(gb_lv):.2f}", flush=True)

    e_gt_endpt = _gt_loss_at(np.asarray(xs_np), sigma, t_grid,
                              math.log(float(e['ls'])),
                              math.log(float(e['var'])))
    s_gt_endpt = _gt_loss_at(np.asarray(xs_np), sigma, t_grid,
                              math.log(float(sp['ls'])),
                              math.log(float(sp['var'])))

    # Per-iter own-objective loss curves
    e_loss_curve = np.array([
        _efgp_loss_at(e_grid, e_top, e_zr,
                       math.log(float(e['ls_path'][t])),
                       math.log(float(e['var_path'][t])))
        for t in range(len(e['ls_path']))])
    s_loss_curve = np.array([
        float(neg_elbo_fn(
            jnp.asarray(math.log(float(sp['ls_path'][t])), dtype=jnp.float32),
            jnp.asarray(math.log(float(sp['var_path'][t])), dtype=jnp.float32)))
        for t in range(len(sp['ls_path']))])

    landscapes.append(dict(
        cfg=cfg, L_efgp=L_efgp, L_efgp_min=L_efgp_min,
        eb_ll=eb_ll, eb_lv=eb_lv,
        L_sp=L_sp, L_sp_min=L_sp_min,
        sb_ll=sb_ll, sb_lv=sb_lv,
        L_gt=L_gt, L_gt_min=L_gt_min,
        gb_ll=gb_ll, gb_lv=gb_lv,
        e_gt_endpt=e_gt_endpt, s_gt_endpt=s_gt_endpt,
        e_loss_curve=e_loss_curve, s_loss_curve=s_loss_curve,
    ))


# ---- 12-panel plot (4 rows × 3 cols) ----
fig, axes = plt.subplots(4, 3, figsize=(20, 22))
for col, (lsd, res) in enumerate(zip(landscapes, all_results)):
    cfg = res['cfg']; e = res['efgp']; sp = res['sp']
    LS_INIT = 1.5
    SP_INIT_LS = 1.5 if cfg['kind'] == 'linear' else 1.0
    e_ls_full = np.concatenate([[LS_INIT], e['ls_path']])
    e_var_full = np.concatenate([[1.0], e['var_path']])
    s_ls_full = np.concatenate([[SP_INIT_LS], sp['ls_path']])
    s_var_full = np.concatenate([[1.0], sp['var_path']])

    # Row 0: GT -log p(x_true; θ) — smoother-free oracle
    L_norm_gt = lsd['L_gt'] - lsd['L_gt_min']
    e_gap = lsd['e_gt_endpt'] - lsd['L_gt_min']
    s_gap = lsd['s_gt_endpt'] - lsd['L_gt_min']
    _render_panel(
        axes[0, col], LOG_LS_GRID, LOG_VAR_GRID, L_norm_gt,
        e_ls_full, e_var_full, s_ls_full, s_var_full,
        lsd['gb_ll'], lsd['gb_lv'],
        title=f"{cfg['name']}\nGround-truth −log p(x_true; θ)  [smoother-free oracle]",
        min_label=f"GT MLE: ℓ={math.exp(lsd['gb_ll']):.2f}, "
                  f"σ²={math.exp(lsd['gb_lv']):.2f}",
        ls_init=LS_INIT, var_init=1.0,
        e_wall=e['wall'], s_wall=sp['wall'],
        e_lat=e['latent_rmse'], s_lat=sp['latent_rmse'],
        e_endpt=e_gap, s_endpt=s_gap)
    if col == 0: axes[0, col].set_ylabel('log σ²', fontsize=10)

    # Row 1: EFGP collapsed loss surface
    L_norm_e = lsd['L_efgp'] - lsd['L_efgp_min']
    _render_panel(
        axes[1, col], LOG_LS_GRID, LOG_VAR_GRID, L_norm_e,
        e_ls_full, e_var_full, s_ls_full, s_var_full,
        lsd['eb_ll'], lsd['eb_lv'],
        title='EFGP collapsed loss surface',
        min_label=f"EFGP min: ℓ={math.exp(lsd['eb_ll']):.2f}, "
                  f"σ²={math.exp(lsd['eb_lv']):.2f}",
        ls_init=LS_INIT, var_init=1.0)
    if col == 0: axes[1, col].set_ylabel('log σ²', fontsize=10)

    # Row 2: SparseGP -ELBO surface
    L_norm_s = lsd['L_sp'] - lsd['L_sp_min']
    _render_panel(
        axes[2, col], LOG_LS_GRID, LOG_VAR_GRID, L_norm_s,
        e_ls_full, e_var_full, s_ls_full, s_var_full,
        lsd['sb_ll'], lsd['sb_lv'],
        title='SparseGP −ELBO surface',
        min_label=f"−ELBO min: ℓ={math.exp(lsd['sb_ll']):.2f}, "
                  f"σ²={math.exp(lsd['sb_lv']):.2f}",
        ls_init=LS_INIT, var_init=1.0)
    if col == 0: axes[2, col].set_ylabel('log σ²', fontsize=10)

    # Row 3: per-iter own-objective curves
    ax = axes[3, col]
    iters_e = np.arange(len(lsd['e_loss_curve']))
    iters_s = np.arange(len(lsd['s_loss_curve']))
    ax.plot(iters_e, lsd['e_loss_curve'], '-o', color='C0',
            markersize=3, linewidth=1.5,
            label='EFGP: collapsed loss along EFGP traj')
    ax2 = ax.twinx()
    ax2.plot(iters_s, lsd['s_loss_curve'], '-s', color='C3',
              markersize=3, linewidth=1.5,
              label='SparseGP: −ELBO along SparseGP traj')
    ax.set_xlabel('EM iter', fontsize=9)
    if col == 0: ax.set_ylabel('EFGP collapsed loss', fontsize=9, color='C0')
    ax.tick_params(axis='y', labelcolor='C0')
    ax2.tick_params(axis='y', labelcolor='C3')
    if col == 2: ax2.set_ylabel('SparseGP −ELBO', fontsize=9, color='C3')
    ax.set_title('per-iter own-objective loss', fontsize=10)
    l1, lab1 = ax.get_legend_handles_labels()
    l2, lab2 = ax2.get_legend_handles_labels()
    ax.legend(l1 + l2, lab1 + lab2, loc='best', fontsize=7)
    ax.grid(alpha=0.3)

fig.suptitle('Hyper trajectories vs three candidate "objective surfaces"\n'
             'row 0 (oracle): −log p(x_true; θ)   |   '
             'row 1: EFGP collapsed loss   |   '
             'row 2: SparseGP −ELBO   |   '
             'row 3: per-iter own-loss', fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

# ---- Summary table ----
print('\nSummary (latent RMSE = Procrustes-aligned vs ground-truth xs):')
print(f"  {'benchmark':30s}  {'EFGP wall':>10s}  {'SP wall':>9s}  "
      f"{'speedup':>8s}  {'EFGP lat_rmse':>14s}  {'SP lat_rmse':>14s}")
for res in all_results:
    e = res['efgp']; sp = res['sp']
    print(f"  {res['cfg']['name']:30s}  {e['wall']:>9.1f}s  "
          f"{sp['wall']:>8.1f}s  {sp['wall']/e['wall']:>7.2f}x  "
          f"{e['latent_rmse']:>14.4f}  {sp['latent_rmse']:>14.4f}")


## Summary: relative drift RMSE vs wall time

This final scatter is now based on the updated per-row protocols. It is useful for within-row comparison, but do not over-interpret cross-row hyper differences as if the three rows were using identical optimization conditions.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

exp_colors = ['C0', 'C3', 'C8']
from matplotlib.lines import Line2D

for i, res in enumerate(all_results):
    efgp, sp = res['efgp'], res['sp']
    name = res['cfg']['name']
    col = exp_colors[i]

    xe, ye = efgp['wall'], efgp['rel']
    xs_, ys_ = sp['wall'], sp['rel']

    ax.scatter(xe, ye, color=col, marker='o', s=110, zorder=5,
               edgecolors='k', linewidths=0.5)
    ax.scatter(xs_, ys_, color=col, marker='s', s=110, zorder=5,
               edgecolors='k', linewidths=0.5)
    ax.plot([xe, xs_], [ye, ys_], color=col, lw=1.2, ls='--', alpha=0.5)

    ha_e = 'right' if xe > xs_ else 'left'
    ha_s = 'left' if xe > xs_ else 'right'
    ax.annotate(f'EFGP\n{name}', (xe, ye), fontsize=7.5, ha=ha_e, va='bottom',
                xytext=(-6 if ha_e == 'right' else 6, 4),
                textcoords='offset points')
    ax.annotate(f'SparseGP\n{name}', (xs_, ys_), fontsize=7.5, ha=ha_s, va='top',
                xytext=(6 if ha_s == 'left' else -6, -4),
                textcoords='offset points')

legend_elems = [
    Line2D([0], [0], marker='o', color='gray', markersize=8, ls='', label='EFGP'),
    Line2D([0], [0], marker='s', color='gray', markersize=8, ls='', label='SparseGP'),
] + [Line2D([0], [0], color=exp_colors[i], lw=2.5,
            label=all_results[i]['cfg']['name'])
     for i in range(len(all_results))]
ax.legend(handles=legend_elems, fontsize=8, loc='upper left', framealpha=0.9)

ax.set_xlabel('Wall time (s)', fontsize=12)
ax.set_ylabel('Relative drift RMSE  (÷ zero-drift baseline)', fontsize=12)
ax.set_title('Speed vs accuracy under the updated stable protocols', fontsize=12)
ax.grid(alpha=0.2)
plt.show()

## Persistent issues

A few real caveats remain, but the earlier notebook failures are now mostly resolved:

- The old EFGP anharmonic failure was largely a support/grid issue. With the broader support box and `eps=1e-3`, the jit-friendly fixed-`K` path is stable and lands close to SparseGP on that row.
- The older hardcoded `K=10` lattice left a real approximation gap on the linear row. Using the initial-tailored fixed lattice (`K=5` here) removes most of that gap while staying fully jit-friendly.
- A tailored-grid oracle is still useful diagnostically: it shows that anharmonic is already well captured by the fixed-`K` path, while linear remains the most grid-sensitive regime.
- On the nonlinear rows, raw hypers remain less identifiable than latent/drift metrics. Different `(ℓ, σ²)` combinations can produce very similar latent and drift quality.
- SparseGP remains more sensitive to aggressive shared schedules. On the linear row, `2e-3` is the strongest stable constant M-step lr; on the nonlinear rows, the conservative SING-demo protocol is still required to avoid collapse or `NaN`s.
- The previous Hutchinson trace estimator was too high-variance on this problem (sign-changing `dws/dlogℓ` gives substantial off-diagonal mass in `A⁻¹ ∂A`).  Switching the M-step to dense Cholesky resolved this — see the M-step landscape plots above for evidence that EFGP now converges directly to the loss minimum across all 3 benchmarks.
- Cross-row wall times remain informative, but they are not from one universal protocol; they are from the current stable protocol for each regime.